<h2>Modelling mass concentration of chlorophyll a in sea water (mg/m³) using data from the CMEMS Data Store</h2>

<hr>
<h4><strong>Data Used</strong></h4>
<ul>
    <li>Copernicus Marine Data</li><br>

| Dataset | CMEMS product ID| CMEMS product<br>description | Variable | Product User Manual |
|:--------------------:|:-----------------------:|:-------------:|:-----------------:|:-----------------:|
| Baltic Sea Multiyear Ocean Colour Plankton,<br>Reflectances and Transparency L3 daily observations | OCEANCOLOUR_BAL_BGC_L3_MY_009_133 | <a href="https://data.marine.copernicus.eu/product/OCEANCOLOUR_BAL_BGC_L3_MY_009_133/description" target="_blank">Description</a> | Chlorophyll a concentration (CHL-a) mg/m³ | <a href="https://documentation.marine.copernicus.eu/PUM/CMEMS-OC-PUM.pdf" target="_blank">User Manual</a> |
  </ul>
 <ul>   
   <li>Coastal water areas (KVF)

| Dataset | Orgination | Metadata |
|:--------------------:|:-----------------------:|:-------------:|
| VM Vattenförekomster kustvatten 2016-2021 |<a href="https://viss.lansstyrelsen.se/" target="_blank">VISS </a>|<a href="https://ext-geodatakatalog.lansstyrelsen.se/GeodataKatalogen/srv/api/records/GetMetaDataById?id=ec8c278f-37a5-4fb3-971f-2920ab22609f" target="_blank">Link</a>|
</ul>

<hr>
<h4><strong>Learning Outcomes</strong></h4>

At the end of this notebook you will be able to:
<li> Set up the environment - install and activate the correct Conda enivironment (wekolab) and install required Pyhon libararies
<li> Connect to the Copernicus Marine Service
<li> Load and visualize mass concentration of chlorophyll a in sea water (mg/m³)
<li> Handle missing data due to cloud coverage (gap-filling)
<li> Generate a summary dataset (mean Chl-a per pixel) 
<li> Read and process coastal water areas (KVF) from shapefiles
<li> Analyze Chl-a within coastal KVF areas
<li> Classify ecological status using EK classification
<li> Visualize and summarize classification results
    
<hr>
<h4><strong>Outline</strong></h4>
Chlorophyll a (Chl-a) is a key indicator of phytoplankton biomass and is widely used as a proxy for assessing the trophic status and ecological health of aquatic ecosystems. In the context of Swedish environmental quality standards, Chl-a concentrations are one of the primary biological quality elements used to classify the ecological status of coastal water bodies. Elevated levels of Chl-a can indicate eutrophication, often driven by nutrient enrichment from land-based sources, and are associated with reduced water clarity. Importantly, Chl-a is also a primary indicator of algal blooms, including harmful algal blooms (HABs), which can lead to oxygen depletion, toxin production, and negative impacts on fisheries, tourism, and public health. Monitoring Chl-a levels at high temporal and spatial resolution is therefore essential for early detection and management of bloom events. 

<hr>
<h4><strong>References</strong></h4>
<ul>
<li>Brando, V.E.; Sammartino, M; Colella, S.; Bracaglia, M.; Di Cicco, A; D’Alimonte, D.; Kajiyama, T., Kaitala, S., Attila, J., 2021b (accepted). Phytoplankton Bloom Dynamics in the Baltic Sea Using a Consistently Reprocessed Time Series of Multi-Sensor Reflectance and Novel Chlorophyll-a Retrievals. Remote Sens. 2021, 13, x.</li>
</ul>

<hr>

<h4><strong>Contents</strong></h4>

<ol>   
<li>Access the parameter for mass concentration of chlorophyll a (mg/m³) in sea water</li>
<li>Read in the coast water areas (KVF)</li>
<li>Mass concentration of chlorophyll a (Chl-a) within coastal water areas (KVF)</li>
<li>EK Classification of chlorophyll a (Chl-a) in Coastal Water Areas (KVF)</li>
</ol>
<hr>

<div class="alert alert-block alert-warning"> 
<h4>Prerequisites:</h4>
<ol>
<li><strong><em>Wekeo account: </em></strong> Create an account via "Register" here: <em>https://wekeo.copernicus.eu/</em></li>
<li><strong><em>Copernicus Marine account: </em></strong> Create an account here: <em>https://data.marine.copernicus.eu/register</em></li>
<li><strong><em>DTO_conda </em></strong> python environment installed and activated as the active kernel (please see README.md for instructions)</li>
<li><strong><em>OR </em></strong> necessary external packages installed in the "wekeolab" environment through the following steps:</li>
<ul>
<li>open a terminal in the wekeolab JupyterHub environment</li>
<li>activate the wekeolab environment <code style="background:red;color:white">conda activate wekeolab</li>
<li>execute the following commands to install the required packages in the wekeolab environment:</li>
        - <code style="background:red;color:white">conda install -c conda-forge xcube bottleneck hvplot geoviews copernicusmarine -y</code><br>
</div>
<hr>

<div class="alert alert-block alert-warning"> 
We start by importing all libraries that will be used in this notebook. This notebook can be run in a Wekeolab Jupyter workspace after installing the extra libraries listed in "Prerequisites." An environment file is also included in this repository for users who would prefer to build their own environment.
</div>

In [ ]:
import pyproj                        
print(pyproj.datadir.get_data_dir()) 
 
# Copernicus Marine Toolbox data import
import copernicusmarine as cm        
 
# File handling and data science
import os                        
import numpy as np                  
import xarray as xr              
import bottleneck                 
import datetime                 
 
# Geospatial data management
import shapely                  
import geopandas as gpd          
import rioxarray                
from xcube.core.gridmapping import GridMapping   
from xcube.core.resampling import resample_in_space 
 
# Visualisations
import matplotlib.pyplot as plt 
import matplotlib.ticker as mticker 
import cartopy.crs as ccrs          
import cartopy.io.shapereader as shpreader 
import cartopy.feature as cfeature 
import hvplot.xarray
import panel as pn
pn.extension()
import hvplot.pandas
from matplotlib.colors import ListedColormap

# Other
from IPython.display import JSON
import warnings
warnings.filterwarnings('ignore')
from shapely.geometry import Polygon
from collections import Counter
import matplotlib.patches as mpatches

<div class="alert alert-info" role="alert">

## <a id='section1'></a>1. Access the parameter for mass concentration of chlorophyll a (mg/m³) in sea water
[Back to top](#TOC_TOP)

</div>

<div class="alert alert-block alert-warning"> 
We first need to provide our username and password from our free account on the Copernicus Marine website:
</div>

In [ ]:
user = "hjonsson"
pwd = "Vargtass1029!!"
cred_path = os.path.expanduser("~/.copernicusmarine/.copernicusmarine-credentials")
if os.path.exists(cred_path):
    os.remove(cred_path)

cm.login(user, pwd)

<div class="alert alert-block alert-warning"> 
We are now ready to remotely access a data cube time series for mass concentration of chlorophyll a in sea water (Chl-a) during the summer months (May 1-September 30 2024). The mass concentration of chlorophyll a in sea water product in OLCI is derived using the method of Brando et al., 2021. The final Chl-a estimate is computed as weighted average;

$$Chl-a_{ENS3} = \frac{\sum_{i=1}^{N=3}w_i \cdot (Chl - a_{MLP})}{\sum_{i=1}^{N=3} w_i}$$

If you would like to change the geographic area please adjust the lat/long supplied in the "bbox" variable. If you would like to change the time scale please change the variables for "start" and "end" dates. After the data is loaded we take a look at the resulting data cube. We see that we have a 3D dataset comprised of 153 days within a rectangular area. The only variable we have chosen to access at this time is Chl-a. Note that the time series data cube is loaded in as a Dask array so that the full dataset is not loaded into server memory until data is manipulated in some way. This allows for much faster data loading, but if you attempt to access a very long time series the server can still be overloaded and crash!
</div>

In [ ]:
ds = cm.open_dataset(
    dataset_id = 'cmems_obs-oc_bal_bgc-plankton_my_l3-olci-300m_P1D',
    minimum_longitude=9.5,
    maximum_longitude=13,
    minimum_latitude=55,
    maximum_latitude=60,
    # enter start and end dates
    start_datetime = "2024-05-01",
    end_datetime = "2024-09-30",
    variables = ['CHL']
)

In [ ]:
# Read in some map background features

land = shpreader.natural_earth('10m', 'cultural', 'admin_0_countries')
land = cfeature.ShapelyFeature(shpreader.Reader(land).geometries(), ccrs.PlateCarree(), edgecolor='black', facecolor='lightgray', lw=0.2)

ocean = shpreader.natural_earth('10m', 'physical', 'ocean')
ocean = cfeature.ShapelyFeature(shpreader.Reader(ocean).geometries(), ccrs.PlateCarree(), edgecolor='none', facecolor='lightblue')

<div class="alert alert-block alert-warning"> 
Now we will visualize the first 4 days of our data. You will see that we have lots of gaps - this is due to cloud cover interfering with the OLCI sensor data acquisition. We will address this problem in the following code.
</div>

In [ ]:
fig = ds.CHL.isel(time=slice(0,4)).plot(
    transform=ccrs.PlateCarree(),
    col='time',
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    vmin=0, 
    vmax=5,
    figsize=(15,5),
    zorder=2
)

[ax.add_feature(cfeature.LAND,zorder=1) for ax in fig.axs.flatten()]
[ax.add_feature(cfeature.OCEAN,zorder=0) for ax in fig.axs.flatten()]

for ax in fig.axs.flat:
    g1 = ax.gridlines(color='black', linewidth=0.5, linestyle='--')
    g1.xlocator = mticker.FixedLocator(np.arange(10, 14, 1))
    g1.ylocator = mticker.FixedLocator(np.arange(55, 60, 1))

<div class="alert alert-block alert-warning"> 
We now apply a forward filling technique to fill in data gaps caused by cloud cover. This takes the last valid pixel value and carries that data through in the time dimension. We limit the forward filling to 7 days to make sure that we are not looking at data outdated by more that a week. We then plot the first four days of data again to see the effect of the forward fill in comparison with the above maps which visualize the raw data. We now have a more complete dataset for mass concentration (mg/m³) of chlorophyll a in sea water (Chl-a) over our entire time series. We also calculate the mean for the dataset (ds_mean), which will be used later in the analysis.
</div>

In [ ]:
ds_fill = ds.ffill(dim='time', limit=7)


<div class="alert alert-block alert-warning"> 
Now we will visualize the first 4 days of our data. You will see that we have lots of gaps - this is due to cloud cover interfering with the OLCI sensor data acquisition. We will address this problem in the following code.
</div>

In [ ]:
fig = ds_fill.CHL.isel(time=slice(0,4)).plot(
    transform=ccrs.PlateCarree(),
    col='time',
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    vmin=0, 
    vmax=5,
    figsize=(15,5),
    zorder=2
)

[ax.add_feature(cfeature.LAND,zorder=1) for ax in fig.axs.flatten()]
[ax.add_feature(cfeature.OCEAN,zorder=0) for ax in fig.axs.flatten()]

for ax in fig.axs.flat:
    g1 = ax.gridlines(color='black', linewidth=0.5, linestyle='--')
    g1.xlocator = mticker.FixedLocator(np.arange(10, 14, 1))
    g1.ylocator = mticker.FixedLocator(np.arange(55, 60, 1))

<div class="alert alert-block alert-warning"> 
We now calculate the average value for each pixel (ds_mean) over the specified time period. This averaged dataset will be used further in the analysis.
</div>

In [ ]:
# remote Time dimension and average for each pixel across time series
ds_mean = ds_fill.mean(dim='time', skipna=True)

<div class="alert alert-info" role="alert">

## <a id='section1'></a>2. Read in the coast water areas (KVF)
[Back to top](#TOC_TOP)

</div>

<div class="alert alert-block alert-warning"> 
Now we load all coastal water body (KVF) areas, which is saved in the external data folder, for the west coast in order to clip the dataset according to the KVF areas. 
</div>

In [ ]:
# read in kustvattenförekomster and clip to the study area
kvf = os.path.join('external_data', 'VM', 'vm.Kustvatten_Vattenforekomster_2016_1.shp')
kvf = gpd.read_file(kvf)
kvf = kvf.to_crs(4326)

c =  {'NW': [9.5, 60],
      'NE': [13, 60],
      'SW': [9.5, 55],
      'SE': [13, 55]}

bb = Polygon([c['NW'], c['NE'], c['SE'], c['SW']])

kvf_clipped = kvf.clip(bb)
kvf_clipped.plot()

<div class="alert alert-block alert-warning"> 
We dissolve kvf by 'TYPOMRKUST' to find all coastal water areas that we are intrested in. 
</div>

In [ ]:
kvf_dissolved = kvf_clipped.dissolve(by='TYPOMRKUST')
kvf_dissolved = kvf_dissolved.reset_index()

for row in kvf_dissolved.itertuples():
    print(f'Index {row.Index}: TYPOMRKUST <{row.TYPOMRKUST}>')


<div class="alert alert-block alert-warning"> 
Now we clip the mean value dataset after the costal water areas (KVF) and store each area in ds_mclipped.
</div>

In [ ]:
ds_mean = ds_mean.rio.write_crs(4326)
ds_mclipped=[]
for index, row in kvf_dissolved.iterrows():
    aoi = ds_mean.rio.clip([row['geometry']], all_touched=True)
    ds_mclipped.append(aoi)


<div class="alert alert-info" role="alert">

## <a id='section1'></a>3. Mass concentration of chlorophyll a (Chl-a) within coastal water areas (KVF)
[Back to top](#TOC_TOP)

</div>

<div class="alert alert-block alert-warning"> 
This step visualizes the spatial distribution of chlorophyll a (Chl-a) concentrations in coastal water areas (KVF). Each map shows the Chl-a concentration for the selected time period and within the coastal water area (KVF).
</div>

In [ ]:
for ind in range(len(ds_mclipped)):
    data = ds_mclipped[ind]
    fig, ax = plt.subplots(1,1, figsize=(8, 6),subplot_kw=dict(projection=ccrs.PlateCarree()))
    cbar = data['CHL'].plot(ax=ax,vmin=0, vmax=5, zorder=2)
    cbar.colorbar.set_label('Chl-a (mg/m³)')

    kvf_dissolved.iloc[[ind]].boundary.plot(ax=ax, edgecolor="black", lw=0.2)

    try:
        area_name = kvf_dissolved.iloc[ind]['TYPOMRKUST']
    except:
        area_name = f"Area {ind}"

    ax.add_feature(cfeature.LAND, zorder=0)    
    ax.add_feature(cfeature.OCEAN, zorder=0)
    
    bounds=kvf_dissolved.iloc[ind].geometry.bounds
    minx, miny, maxx, maxy = bounds
    pad_x = (maxx - minx) * 0.1
    pad_y = (maxy - miny) * 0.1
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    
    ax.set_title(f"Chlorophyll a (Chl-a) in sea water\n KVF:{area_name}")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    plt.show()

<div class="alert alert-block alert-warning"> 
Now we visualizes the mass concentration of chlorophyll a in sea water (mg/m³) in all coastal water areas (KVF). The map shows Chl-a concentration for the selected time period and within the coastal water area (KVF).
</div>

In [ ]:
# Create figure and axis with map projection
fig, ax = plt.subplots(1, 1, figsize=(12, 10), subplot_kw=dict(projection=ccrs.PlateCarree()))

# Plot CHL data from each KVF on the same map
for ind in range(len(ds_mclipped)):
    data = ds_mclipped[ind]
    chl_plot = data['CHL'].plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        vmin=0,
        vmax=5,
        add_colorbar=False,
        zorder=1
    )

# Add KVF labels
for idx, row in kvf_dissolved.iterrows():
    centroid = row.geometry.centroid
    label = str(row['TYPOMRKUST'])
    ax.text(
        centroid.x, centroid.y, label,
        fontsize=10, ha='center', va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='black', alpha=0.7)
    )

# Add colorbar
cbar = fig.colorbar(chl_plot, ax=ax, orientation='vertical', shrink=0.7)
cbar.set_label('Chl-a (mg/m³)', fontsize=12)

# Plot the boundaries of all KVF areas
kvf_dissolved.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5, zorder=2)

# Add land and ocean background
ax.add_feature(cfeature.LAND, zorder=0)
ax.add_feature(cfeature.OCEAN, zorder=0)

# Set map extent based on all KVF areas
minx, miny, maxx, maxy = kvf_dissolved.total_bounds
pad_x = (maxx - minx) * 0.05
pad_y = (maxy - miny) * 0.05
ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

# Add title and axis labels
ax.set_title("Chlorophyll a (Chl-a) concentration in all coastal water areas (KVF)", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect('equal')
plt.tight_layout()

plt.show()

<div class="alert alert-block alert-warning"> 
This step creates an interactive map showing the mass concentration of chlorophyll a in sea water (mg/m³) in all coastal water areas (KVF) along the Swedish west coast. The Chl-a data from all KVF areas are combined into a single map, like the map in the previous cell. <b>Note:</b>  When publishing the notebook to GitHub, the map will not be visible due to large data size. However, the code is correct, and the interactive map will appear when the notebook is run locally or in an Jupyter environment.
</div>


In [ ]:
# Combine CHL-layers from all KVF datasets
chl_layers = []

for ds in ds_mclipped:
    chl = ds['CHL'].sortby(['latitude', 'longitude'])  # Ensure monotonic order
    chl_layers.append(chl)

# Combine into 3D array, average over "layer" dimension
combined_chl = xr.concat(chl_layers, dim="layer").mean(dim="layer", skipna=True)

# Plot Chl-a
chl_plot = combined_chl.hvplot.quadmesh(
    x='longitude',
    y='latitude',
    cmap='viridis',
    clim=(0, 5),
    colorbar=True,
    tiles=True,
    frame_height=600,
    frame_width=700,
    alpha=0.9,
    title="Chlorophyll a (Chl-a) concentration in all coastal water areas (KVF)"
)
# Plot KVF boundaries
boundary_plot = kvf_dissolved.hvplot(
    line_width=1,
    color='black',
    fill_alpha=0,
    line_alpha=1,
    tools=[],
    hover_cols=[],
    title=None,
    geo=True
)
#Combine all layers into one plot
final_plot = chl_plot * boundary_plot
final_plot

<div class="alert alert-info" role="alert">

## <a id='section1'></a>4. EK Classification of chlorophyll a (Chl-a) in Coastal Water Areas (KVF)
[Back to top](#TOC_TOP)

</div>

<div class="alert alert-block alert-warning"> 
This step processes chlorophyll a measurements from coastal water bodies to classify their ecological status. It utilizes thresholds from the HVMFS 2019:25 guidelines, specifically the reference values and ecological quality ratios (EK), to determine the status classes: High, Good, Moderate, Unsatisfactory, and Poor. The Ecological Quality Ratio - refered to in Swedish as Ekologisk Kvot (EK) - calculated as: 
$$EK = \left( \frac{\text{Reference Value}}{\text{Measured Value}} \right)$$
    
The result is a series of maps, one per coastal water body, illustrating the EK classification. For each area, a summary table is also provided, showing the pixel count and percentage distrubution for each class within the area.
</div>

In [ ]:
# EK-table. A list containing reference values (Rv) and threshold ratios (HG, GM, MO, OD) for different water body types (Typ), as specified in HVMFS 2019:25.
ek_table = [
    {"Typ": "1n", "Rv": 1.15,   "HG": 0.76, "GM": 0.62, "MO": 0.35, "OD": 0.19},
    {"Typ": "1s", "Rv": 1.6,    "HG": 0.76, "GM": 0.57, "MO": 0.35, "OD": 0.2},
    {"Typ": "2", "Rv": 1.37,    "HG": 0.79, "GM": 0.53, "MO": 0.34, "OD": 0.23},
    {"Typ": "3",  "Rv": 0.99,   "HG": 0.79, "GM": 0.63, "MO": 0.31, "OD": 0.18},
    {"Typ": "25", "Rv": 1.8,    "HG": 0.86, "GM": 0.67, "MO": 0.44, "OD": 0.28},
    {"Typ": "4",  "Rv": 1.0,    "HG": 0.83, "GM": 0.67, "MO": 0.33, "OD": 0.17},
    {"Typ": "5",  "Rv": 0.99,   "HG": 0.83, "GM": 0.67, "MO": 0.33, "OD": 0.17},
    {"Typ": "6",  "Rv": 0.94,   "HG": 0.82, "GM": 0.59, "MO": 0.37, "OD": 0.18},
    {"Typ": "7",  "Rv": 1.3,    "HG": 0.83, "GM": 0.70, "MO": 0.40, "OD": 0.20},
]

# Dictionary mapping each water body type to its corresponding thresholds
ek_dict = {row["Typ"]: row for row in ek_table}

# Dictionary assigning specific colors to each ecological status class 
class_colors = {
    "High": "green",
    "Good": "lightgreen",
    "Moderate": "orange",
    "Unsatisfactory": "orangered",
    "Poor": "red"
}

# Maps each status class to an integer
class_to_int = {k: i for i, k in enumerate(class_colors.keys())}
int_to_class = list(class_colors.keys())

# create dataArray for the combined classification result
combined_array = None


""" Process each water areas data and calculates the EK ratio by dividing the reference value by the Chlorophyll a value, 
   and assigns an ecological status class based on the thresholds.The results are stored in the DataArray combined_array."""

for ind in range(len(ds_mclipped)):
    data = ds_mclipped[ind]

    try:
        typ = str(kvf_dissolved.iloc[ind]['TYPOMRKUST']).strip()
        thresholds = ek_dict[typ]
        ref = thresholds["Rv"]
    except KeyError:
        continue

    chl_a_values = data['CHL']
    ek_values = ref / chl_a_values
    valid_mask = ~np.isnan(chl_a_values)

    classification_array = np.full_like(chl_a_values, np.nan)
    mask = valid_mask.copy()
    classification_array[(ek_values > thresholds["HG"]) & mask] = class_to_int["High"]
    mask &= ~(ek_values > thresholds["HG"])
    classification_array[(ek_values > thresholds["GM"]) & mask] = class_to_int["Good"]
    mask &= ~(ek_values > thresholds["GM"])
    classification_array[(ek_values > thresholds["MO"]) & mask] = class_to_int["Moderate"]
    mask &= ~(ek_values > thresholds["MO"])
    classification_array[(ek_values > thresholds["OD"]) & mask] = class_to_int["Unsatisfactory"]
    mask &= ~(ek_values > thresholds["OD"])
    classification_array[mask] = class_to_int["Poor"]

    class_da = xr.DataArray(
        classification_array,
        coords=data['CHL'].coords,
        dims=data['CHL'].dims
    )
    
    # Combine all classifications
    if combined_array is None:
        combined_array = class_da
    else:
        combined_array = combined_array.combine_first(class_da)

    #Set Up Colormap and Normalization and ensuring that missing data is transparent
    cmap = ListedColormap([class_colors[c] for c in int_to_class])
    cmap.set_bad((0,0,0,0))
    norm = plt.Normalize(vmin=0, vmax=len(class_to_int) - 1)

    try:
        area_name = kvf_dissolved.iloc[ind]['TYPOMRKUST']
    except:
        area_name = f"Area {ind}"
        
    #Create the plot
    fig, ax = plt.subplots(1,1, figsize=(10, 8),subplot_kw=dict(projection=ccrs.PlateCarree()))
    class_da.plot(ax=ax, cmap=cmap, norm=norm, add_colorbar=False)
    kvf_dissolved.iloc[[ind]].plot(ax=ax, edgecolor='black', facecolor='none', lw=0.6)
    
    #Add Legend and Additional Features
    legend_patches = [mpatches.Patch(color=class_colors[label], label=label)
                  for label in int_to_class
                 ]
    ax.legend(handles=legend_patches, loc='lower left', title="EK class", frameon=True, facecolor='white')
    
    bounds=kvf_dissolved.iloc[ind].geometry.bounds
    minx, miny, maxx, maxy = bounds
    pad_x = (maxx - minx) * 0.4
    pad_y = (maxy - miny) * 0.4
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    
    ax.add_feature(cfeature.LAND, zorder=0)    
    ax.add_feature(cfeature.OCEAN, zorder=0) 
    ax.set_title(f"EK classification (Chl-a)\n KVF:{area_name}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()
    

    # Flatten the classification array and exclude NaN values
    flat_array = classification_array[~np.isnan(classification_array)].astype(int).ravel()
    
    # Count number of pixels per class
    counts = Counter(flat_array)

    #Define the order of classes
    ordered_classes = ["High", "Good", "Moderate", "Unsatisfactory", "Poor"]
    
    # Create list of rows with class name, pixel count, and percentage
    pixel_stats = []
    total_pixels = len(flat_array)

    plt.show()
    
    for class_name in ordered_classes:
        class_index = class_to_int[class_name]
        count = counts.get(class_index, 0)
        percentage = 100 * count / total_pixels if total_pixels > 0 else 0
        pixel_stats.append([class_name, count, round(percentage, 1)])

    # Create and display the statistic table
    pixel_df = pd.DataFrame(pixel_stats, columns=["Class", "Pixel count", "(%)"])
    pixel_df.index = [''] * len(pixel_df) 
    display(pixel_df)

<div class="alert alert-block alert-warning"> 
This step visualizes the combined classification of all coastal water areas (KVF) in a single map. It uses the results from the previous classification step, which assigned each grid cell an ecological status class (High, Good, Moderate, Unsatisfactory, or Poor) based on thresholds from the HVMFS 2019:25 guidelines. The output is one comprehensive map displaying the EK classification for all areas.
</div>

In [ ]:
# Define color map and normalization for Chl-a EK classification
cmap = ListedColormap([class_colors[c] for c in int_to_class])
cmap.set_bad((0, 0, 0, 0))  # Transparent for NaNs
norm = plt.Normalize(vmin=0, vmax=len(int_to_class) - 1)

# Create figure and axis with geographical projection
fig, ax = plt.subplots(1, 1, figsize=(15, 13), subplot_kw=dict(projection=ccrs.PlateCarree()))

# Plot the combined classified chl-a data
img = combined_array.plot(
    ax=ax,
    cmap=cmap,
    norm=norm,
    add_colorbar=False
)

# Add legend
legend_patches = [mpatches.Patch(color=class_colors[label], label=label) for label in int_to_class]
ax.legend(handles=legend_patches, loc='upper right', title="EK class", frameon=True, facecolor='white')

# Plot water body boundaries
kvf_dissolved.plot(ax=ax, edgecolor='black', facecolor='none', linewidth=1)

# Add KVF labels
for idx, row in kvf_dissolved.iterrows():
    centroid = row.geometry.centroid
    label = str(row['TYPOMRKUST'])
    ax.text(
        centroid.x, centroid.y, label,
        fontsize=10, ha='center', va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='black', alpha=0.7)
    )

# Set map extent
minx, miny, maxx, maxy = kvf_dissolved.total_bounds
pad_x = (maxx - minx) * 0.05
pad_y = (maxy - miny) * 0.05
ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

# Add background features
ax.add_feature(cfeature.LAND, zorder=0)
ax.add_feature(cfeature.OCEAN, zorder=0)
ax.set_title("EK classification (Chl-a)", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect('equal')
plt.tight_layout()

plt.show()

<div class="alert alert-block alert-warning"> 
This step creates an interactive map showing the ecological status of all coastal water areas (KVF) based on mass concentration of chlorophyll a, like the map in the previous cell. <b>Note:</b> When publishing the notebook to GitHub, the map will not be visible due to large data size. However, the code is correct, and the interactive map will appear when the notebook is run locally or in an Jupyter environment.
</div>


In [ ]:
# Colormap for EK classes
int_to_class = list(class_colors.keys())
cmap = list(class_colors.values())

# Plot classified map
ek_plot = combined_array.hvplot.quadmesh(
    x='longitude',
    y='latitude',
    cmap=cmap,
    clim=(0, len(class_colors) - 1),
    colorbar=False,
    tiles=True,
    frame_height=600,
    frame_width=700,
    alpha=0.9,
    title="EK classification (Chl-a)",
    legend=True
)

# Plot KVF boundaries
boundary_plot = kvf_dissolved.hvplot(
    line_width=1,
    line_color='black',
    fill_alpha=0,
    tools=[],
    hover_cols=[],
    geo=True
)

# Combine to one interactive map
final_plot = ek_plot * boundary_plot
final_plot